# Experiment 2: Sufficiency Test - Individual Neuron Activation

**Purpose**
- Reproduce Shiu et al. (2024) Figure 1e
- Test which neurons can activate MN9 when individually stimulated
- Method: Activate each of top 200 neurons individually at different frequencies

**Configuration**
- Neurons to test: Top 200 responsive neurons (from Exp1)
- Frequencies: 25, 50, 75, 100, 125, 150, 175, 200 Hz (8 frequencies)
- Trials: 10 per condition
- Batches: 8 batches (1 frequency each, 200 neurons)
- Workers: 4

**Expected Runtime**
- Per batch: 200 neurons ÷ 4 workers × 130s ≈ **1.8 hours**
- Total: 8 batches × 1.8h ≈ **14.4 hours** (can run over 3-4 days)

**Memory Requirements**
- Peak: ~8-9GB (safe for 16GB system)
- Swap: 0GB (verified in Exp3)

**Outputs**
- `results/exp2_sufficiency_test/exp2_complete_results.pkl`
- `results/exp2_sufficiency_test/sufficient_neurons.npy` (for Exp3 Venn)
- `results/exp2_sufficiency_test/figures/` (heatmap, response curves, summary)

**Dependencies**
- Requires: `results/exp1_sugar_activation/top_200_neurons.npy`

---

## 1. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import time
import pickle
import os

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

from brian2 import Hz, ms, start_scope

from joblib import Parallel, delayed
import gc

print("✅ Standard libraries imported")

In [ ]:
# FlyWire tools
from caveclient import CAVEclient

print("Initializing CAVEclient...")
client = CAVEclient('flywire_fafb_public')
print("✅ Connected to FlyWire")

In [ ]:
# FlyLIF modules
from flylif.core.parameters import DEFAULT_PARAMS
from flylif.core.data_loader import load_simulation_data
from flylif.core.experiments import run_exp2_parallel
from flylif.utils.cave_utils import convert_neuron_list_cached
from flylif.utils.checkpoint import CheckpointManager
from flylif.utils.memory_utils import print_memory, check_memory_safe, memory_cleanup, MemoryMonitor

print("✅ FlyLIF modules imported")

In [ ]:
# Auto-reload
%load_ext autoreload
%autoreload 2
print("✅ Auto-reload enabled")

## 2. Configuration

In [ ]:
BASE_DIR = Path.home() / 'mylibrary'/'connectome'/'lifmodel'

CONFIG = {
    'data_dir': BASE_DIR / 'data_783',
    'connections_file': 'proofread_connections_783.feather',
    'root_ids_file': 'proofread_root_ids_783.npy',
}

print("Configuration:")
print(f"  Data dir: {CONFIG['data_dir']}")

## 3. Brian2 Cache Check & Clean

In [ ]:
# ===== Clean Brian2 Cache (Prevent Lock Issues) =====
import shutil

cache_dir = Path.home() / 'Library/Caches/cython/brian_extensions'

if cache_dir.exists():
    # Check lock file age
    locks = list(cache_dir.glob('*.lock'))
    
    if locks:
        oldest_lock_age = max(time() - os.path.getmtime(f) for f in locks)
        
        # If stale locks exist (>1 hour old), clean cache
        if oldest_lock_age > 3600:
            print(f"⚠️  Found {len(locks)} stale locks (oldest: {oldest_lock_age/3600:.1f}h)")
            print("   Cleaning Brian2 cache...")
            shutil.rmtree(cache_dir)
            cache_dir.mkdir(parents=True, exist_ok=True)
            print("   ✅ Cleaned")
        else:
            ages = sorted([time() - os.path.getmtime(f) for f in locks])
            recent = sum(1 for a in ages if a < 600)
            medium = sum(1 for a in ages if 600 <= a < 3600)
            old = sum(1 for a in ages if a >= 3600)
            
            print(f"✓ Brian2 cache status:")
            print(f"  Recent (<10min): {recent}")
            print(f"  Medium (10min-1h): {medium}")
            print(f"  Old (>1h): {old}")
            
            if old == 0:
                print(f"  ✓ All locks healthy")
    else:
        print("✓ Brian2 cache clean (no locks)")
else:
    print("✓ Brian2 cache not found (will be created on first use)")

In [ ]:
cache_dir = Path.home() / 'Library/Caches/cython/brian_extensions'

print("Completely removing Brian2 cache...")
shutil.rmtree(cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)
print("✅ Fresh start!")

# 验证
locks_after = list(cache_dir.glob('*.lock'))
print(f"Lock count after clean: {len(locks_after)} (should be 0)")

## 4. Load Data

In [ ]:
print("\n" + "=" * 70)
print("Loading Optimized Connectivity Data")
print("=" * 70)

print_memory("Before: ")

t0 = time()
DATA = load_simulation_data(CONFIG, verbose=True)

print(f"\n✅ Loaded in {time()-t0:.1f}s")
print(f"   Data size: {DATA['df_conn'].memory_usage(deep=True).sum()/1e6:.0f} MB")

print_memory("After: ")

## 5. Load Neuron IDs

### 5.1 MN9 target neuron

In [ ]:
print("\n" + "=" * 70)
print("Loading Neuron IDs")
print("=" * 70)

# Cache directory
cache_dir = BASE_DIR / 'flylif' / 'cache' / 'id_conversions'

# MN9 (target neuron to monitor)
NEU_MN9_v630 = [720575940660219265]

NEU_MN9_RIGHT = convert_neuron_list_cached(
    old_list=NEU_MN9_v630,
    cache_file=cache_dir / 'mn9_v630_to_v783.pkl',
    new_ver=783,
    verbose=False
)

print(f"\n✅ Loaded MN9 target:")
print(f"  MN9 (right): {NEU_MN9_RIGHT[0]}")

### 5.2 Load Top 200 neurons from Exp1

In [ ]:
# Load Top 200 neurons from Exp1
exp1_dir = BASE_DIR /'lif_simulation' / 'results' / 'exp1_sugar_activation'
top_200_file = exp1_dir / 'top_200_neurons.npy'

if not top_200_file.exists():
    raise FileNotFoundError(
        f"❌ Top 200 neurons file not found: {top_200_file}\n"
        f"   Please run exp1_sugar_activation.ipynb first!"
    )

top_200_neurons = list(np.load(top_200_file))

print(f"\n✅ Loaded Top 200 neurons from Exp1")
print(f"  File: {top_200_file}")
print(f"  Count: {len(top_200_neurons)}")
print(f"\n  First 5: {top_200_neurons[:5]}")

## 6. Experiment Configuration

In [ ]:
# ========== EXPERIMENT PARAMETERS ==========

# Frequencies (matching original paper Fig 1e)
FREQ_LIST_FULL = [25, 50, 75, 100, 125, 150, 175, 200]

# Batch configuration (by frequency)
BATCH_CONFIG = {
    'batch1': [25],   # 200 neurons × 1 freq
    'batch2': [50],
    'batch3': [75],
    'batch4': [100],
    'batch5': [125],
    'batch6': [150],
    'batch7': [175],
    'batch8': [200],
}

N_TRIALS = 10
N_WORKERS = 4

# Checkpoint directory
CHECKPOINT_DIR = './checkpoints/exp2_full'

print("=" * 70)
print("Experiment 2: Sufficiency Test Configuration")
print("=" * 70)
print(f"\n[Configuration]")
print(f"  Neurons to test: {len(top_200_neurons)}")
print(f"  Frequencies: {FREQ_LIST_FULL}")
print(f"  Trials per condition: {N_TRIALS}")
print(f"  Workers: {N_WORKERS}")
print(f"  Checkpoint: {CHECKPOINT_DIR}")

print(f"\n[Batch Strategy]")
print(f"  Total batches: {len(BATCH_CONFIG)}")
print(f"  Tasks per batch: {len(top_200_neurons)} neurons × 1 freq = 200")
print(f"  Time per batch: ~1.5-2 hours")

print(f"\n[Total Workload]")
total_tasks = len(top_200_neurons) * len(FREQ_LIST_FULL)
print(f"  Total simulations: {total_tasks}")
print(f"  Estimated time: ~14-16 hours")

print("\n" + "=" * 70)

## 7. Memory Check

In [ ]:
print("\nPre-experiment memory status:")
is_safe, msg = check_memory_safe(threshold=75.0, swap_threshold=0.5)
print(f"  {msg}")

if not is_safe:
    print("\n⚠️  High memory detected!")
    print("   Recommend: Kernel → Restart Kernel")
    print("   Then re-run Cells 1-7")
else:
    print("\n✅ Memory OK, ready to start")

test

debug for exp3

In [ ]:
print("\n" + "=" * 70)
print("Loading Neuron IDs")
print("=" * 70)

# Cache directory
cache_dir = BASE_DIR / 'flylif' / 'cache' / 'id_conversions'

# Original v630 IDs
NEU_SUGAR_v630 = [
    720575940624963786, 720575940630233916, 720575940637568838, 720575940638202345, 720575940617000768,
    720575940630797113, 720575940632889389, 720575940621754367, 720575940621502051, 720575940640649691,
    720575940639332736, 720575940616885538, 720575940639198653, 720575940620900446, 720575940617937543,
    720575940632425919, 720575940633143833, 720575940612670570, 720575940628853239, 720575940629176663,
    720575940611875570
]

NEU_MN9_v630 = [720575940660219265]

# Convert to v783
NEU_SUGAR_LEFT = convert_neuron_list_cached(
    old_list=NEU_SUGAR_v630,
    cache_file=cache_dir / 'sugar_grns_v630_to_v783.pkl',
    new_ver=783,
    verbose=False
)

NEU_MN9_RIGHT = convert_neuron_list_cached(
    old_list=NEU_MN9_v630,
    cache_file=cache_dir / 'mn9_v630_to_v783.pkl',
    new_ver=783,
    verbose=False
)

print(f"\n✅ Loaded neuron IDs:")
print(f"  Sugar GRNs: {len(NEU_SUGAR_LEFT)}")
print(f"  MN9 (right): {NEU_MN9_RIGHT[0]}")

In [ ]:
# 创建测试worker（激活21个Sugar GRNs + 1个测试神经元）
from flylif.core.network import build_network
from flylif.core.simulation import run_simulation

start_scope()

columns = DATA['columns']
net = build_network(
    data=DATA, pre_col=columns['pre_col'],
    post_col=columns['post_col'], weight_col=columns['weight_col'],
    nt_prob_cols=columns.get('nt_prob_cols', {}),
    params=DEFAULT_PARAMS, syn_threshold=5, verbose=False
)

# 测试：同时激活21个Sugar GRNs（模拟Exp3的输入复杂度）
t0 = time()

result = run_simulation(
    net_components=net,
    neu_exc=NEU_SUGAR_LEFT,  # ← 21个（不是1个）
    params={'r_poi': 50 * Hz},
    n_trials=10,
    verbose=False
)

multi_input_time = time() - t0

print(f"\n21-input simulation time: {multi_input_time:.1f}s")
print(f"vs Exp2 single-input: ~95min/200 = 28.5s/task")
print(f"Ratio: {multi_input_time / 28.5:.1f}×")

In [ ]:
# 测试Exp1的单个frequency task（21输入，全脑记录）
from flylif.core.network import build_network
from flylif.core.simulation import run_simulation

start_scope()
columns = DATA['columns']
net = build_network(
    data=DATA, pre_col=columns['pre_col'],
    post_col=columns['post_col'], weight_col=columns['weight_col'],
    nt_prob_cols=columns.get('nt_prob_cols', {}),
    params=DEFAULT_PARAMS, syn_threshold=5, verbose=False
)

# 模拟Exp1的单个task
t0 = time()

result = run_simulation(
    net_components=net,
    neu_exc=NEU_SUGAR_LEFT,  # 21个
    params={'r_poi': 100 * Hz},
    n_trials=10,
    verbose=False
)

exp1_task_time = time() - t0

print(f"\nExp1 single task time: {exp1_task_time:.1f}s")
print(f"Exp3 single task time: 571s")
print(f"Ratio: {exp1_task_time / 571:.2f}×")

## 8. Execute Batches

**Instructions**:
- Run each batch cell separately
- Each batch takes ~1.5-2 hours
- Can restart kernel between batches if needed
- Checkpoint auto-saves progress (safe to interrupt)

**Strategy**:
- Option A: Run 2 batches per day over 4 days
- Option B: Run overnight batches (2-3 at a time)
- Option C: Parallel with Exp3 (if running both experiments)

### 8.1 Batch 1: 25 Hz

In [ ]:
print("\n" + "=" * 70)
print("Batch 1: Frequency 25 Hz")
print("=" * 70)

with MemoryMonitor(label="Batch1_25Hz", warn_threshold=80.0):
    t0 = time()
    
    results_batch1 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[25],  # Single frequency
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 1 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")

### 8.2 Batch 2: 50 Hz

In [ ]:
# ⚠️  If kernel restarted, re-run Cells 1-7 first!

print("\n" + "=" * 70)
print("Batch 2: Frequency 50 Hz")
print("=" * 70)

with MemoryMonitor(label="Batch2_50Hz"):
    t0 = time()
    
    results_batch2 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[50],
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 2 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")

### 8.3 Batch 3: 75 Hz

In [ ]:
print("\n" + "=" * 70)
print("Batch 3: Frequency 75 Hz")
print("=" * 70)

with MemoryMonitor(label="Batch3_75Hz"):
    t0 = time()
    
    results_batch3 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[75],
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 3 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")
os.system('say "Hey , your 75hz test are done. Please check the memory."')

### 8.4 Batch 4: 100 Hz

In [ ]:
print("\n" + "=" * 70)
print("Batch 4: Frequency 100 Hz")
print("=" * 70)

with MemoryMonitor(label="Batch4_100Hz"):
    t0 = time()
    
    results_batch4 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[100],
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 4 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")

os.system('say "Hey , your 100hz test are done. Please check the memory."')

### 8.5 Batch 5: 125 Hz

In [ ]:
print("\n" + "=" * 70)
print("Batch 5: Frequency 125 Hz")
print("=" * 70)

with MemoryMonitor(label="Batch5_125Hz"):
    t0 = time()
    
    results_batch5 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[125],
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 5 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")
os.system('say "Hey , your 125hz test are done. Please check the memory."')

### 8.6 Batch 6: 150 Hz

In [ ]:
print("\n" + "=" * 70)
print("Batch 6: Frequency 150 Hz")
print("=" * 70)

with MemoryMonitor(label="Batch6_150Hz"):
    t0 = time()
    
    results_batch6 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[150],
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 6 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")
os.system('say "Hey , your 150hz test are done. Please check the memory."')

### 8.7 Batch 7: 175 Hz

In [ ]:
print("\n" + "=" * 70)
print("Batch 7: Frequency 175 Hz")
print("=" * 70)

with MemoryMonitor(label="Batch7_175Hz"):
    t0 = time()
    
    results_batch7 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[175],
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 7 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")
os.system('say "Hey , your 175hz test are done. Please check the memory."')

### 8.8 Batch 8: 200 Hz (Final)

In [ ]:
print("\n" + "=" * 70)
print("Batch 8: Frequency 200 Hz (Final Batch)")
print("=" * 70)

with MemoryMonitor(label="Batch8_200Hz"):
    t0 = time()
    
    results_batch8 = run_exp2_parallel(
        data=DATA,
        neurons_to_test=top_200_neurons,
        freqs=[200],
        target_neurons=NEU_MN9_RIGHT,
        params=DEFAULT_PARAMS,
        n_trials=N_TRIALS,
        n_workers=N_WORKERS,
        checkpoint_dir=CHECKPOINT_DIR,
        verbose=True
    )
    
    elapsed = time() - t0

print(f"\n✅ Batch 8 complete! Time: {elapsed/60:.1f} min")
print_memory("Final: ")

print("\n" + "=" * 70)
print("🎉 All 8 Batches Complete!")
print("=" * 70)
os.system('say "Hey , your 200hz test are done. Please check the memory."')

## 9. Merge All Results

In [ ]:
print("\n" + "=" * 70)
print("Merging All Batch Results")
print("=" * 70)

# Re-run with all frequencies to collect all results from checkpoint
print("\nLoading complete results from checkpoint...")
print("(This will skip all tasks and just load saved data)")

t0 = time()

results_exp2_full = run_exp2_parallel(
    data=DATA,
    neurons_to_test=top_200_neurons,
    freqs=FREQ_LIST_FULL,  # All 8 frequencies
    target_neurons=NEU_MN9_RIGHT,
    params=DEFAULT_PARAMS,
    n_trials=N_TRIALS,
    n_workers=N_WORKERS,
    checkpoint_dir=CHECKPOINT_DIR,
    verbose=False
)

print(f"\n✅ Merged in {time()-t0:.1f}s")

# Verify completeness
print(f"\n[Verification]")
for freq in FREQ_LIST_FULL:
    n_tested = len(results_exp2_full[freq])
    
    status = "✅" if n_tested == 200 else "❌"
    print(f"  {status} {freq} Hz: {n_tested}/200 neurons tested")

## 10. Analysis: Identify Sufficient Neurons

In [ ]:
print("\n" + "=" * 70)
print("Analyzing Sufficient Neurons")
print("=" * 70)

# For each frequency, identify neurons that activate MN9 (firing rate > 0)
sufficient_by_freq = {}

for freq in FREQ_LIST_FULL:
    sufficient_neurons = []
    
    for nid, stats in results_exp2_full[freq].items():
        if stats['mean'] > 0:  # Activates MN9
            sufficient_neurons.append(nid)
    
    sufficient_by_freq[freq] = sufficient_neurons
    
    # Calculate statistics
    mn9_rates = [stats['mean'] for stats in results_exp2_full[freq].values()]
    max_rate = max(mn9_rates)
    mean_rate = np.mean([r for r in mn9_rates if r > 0]) if sufficient_neurons else 0
    
    print(f"\n{freq} Hz:")
    print(f"  Sufficient neurons: {len(sufficient_neurons)}/200 ({100*len(sufficient_neurons)/200:.1f}%)")
    print(f"  Max MN9 activation: {max_rate:.1f} Hz")
    print(f"  Mean MN9 (active): {mean_rate:.1f} Hz")

# Neurons sufficient at ANY frequency
sufficient_any = set()
for neurons in sufficient_by_freq.values():
    sufficient_any.update(neurons)

# Neurons sufficient at HIGHEST frequency (200 Hz)
sufficient_200hz = set(sufficient_by_freq[200])

print(f"\n[Summary]")
print(f"  Sufficient at ANY frequency: {len(sufficient_any)}")
print(f"  Sufficient at 200 Hz: {len(sufficient_200hz)}")

## 11. Visualization

In [ ]:
print("\n" + "=" * 70)
print("Generating Figures")
print("=" * 70)

# Create output directories
output_dir = Path('./results/exp2_sufficiency_test')
fig_dir = output_dir / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

### 11.1 Sufficiency heatmap (Figure 1e style)

In [ ]:
print("\nFigure 1: Sufficiency test heatmap")

from flylif.utils.visualization import plot_response_heatmap

# Convert to format for plot_response_heatmap
all_mn9_rates = {}
for freq in FREQ_LIST_FULL:
    all_mn9_rates[freq] = {
        nid: stats['mean'] 
        for nid, stats in results_exp2_full[freq].items()
    }

# Plot with neuron order from Exp1 (maintain consistency)
fig, ax, _ = plot_response_heatmap(
    all_firing_rates=all_mn9_rates,
    freq_list=FREQ_LIST_FULL,
    neuron_order=top_200_neurons,  # Use Exp1 order
    cmap=sns.color_palette("Spectral_r", as_cmap=True),
    save_path=fig_dir / 'fig1_sufficiency_heatmap.png'
)

# Customize labels for Exp2
ax.set_xlabel('Activation Firing Rate (Hz)', fontsize=12)
ax.set_ylabel('Neurons Tested (ordered by sugar response)', fontsize=12)
ax.set_title('Sufficiency Test: Individual Neuron → MN9 Activation', 
             fontsize=13, fontweight='bold')

plt.tight_layout()
# plt.savefig(fig_dir / 'fig1_sufficiency_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"  ✅ Saved: fig1_sufficiency_heatmap.png")

### 11.2 Summary bar charts

In [ ]:
print("\nFigure 2: Summary statistics")

# Prepare summary data
summary_data = []
for freq in FREQ_LIST_FULL:
    mn9_rates = [stats['mean'] for stats in results_exp2_full[freq].values()]
    n_activate = sum(1 for r in mn9_rates if r > 0)
    max_rate = max(mn9_rates)
    mean_rate = np.mean([r for r in mn9_rates if r > 0]) if n_activate > 0 else 0
    
    summary_data.append({
        'Frequency': freq,
        'N_Activate_MN9': n_activate,
        'Max_MN9_Rate': max_rate,
        'Mean_MN9_Rate': mean_rate,
    })

df_summary = pd.DataFrame(summary_data)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Count of sufficient neurons
ax1 = axes[0]
bars = ax1.bar(df_summary['Frequency'], df_summary['N_Activate_MN9'],
               color='steelblue', edgecolor='black', alpha=0.8, width=12)
ax1.set_xlabel('Activation Frequency (Hz)', fontsize=12)
ax1.set_ylabel('Neurons Activating MN9 (>0 Hz)', fontsize=12)
ax1.set_title('Sufficient Neurons vs Frequency', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}', ha='center', va='bottom', fontsize=10)

# Panel B: Mean MN9 response
ax2 = axes[1]
ax2.bar(df_summary['Frequency'], df_summary['Mean_MN9_Rate'],
        color='coral', edgecolor='black', alpha=0.8, width=12)
ax2.set_xlabel('Activation Frequency (Hz)', fontsize=12)
ax2.set_ylabel('Mean MN9 Firing Rate (Hz)', fontsize=12)
ax2.set_title('Mean MN9 Response (Active Neurons)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
# plt.savefig(fig_dir / 'fig2_summary_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"  ✅ Saved: fig2_summary_statistics.png")

### 11.3 Top 10 neurons response curves

In [ ]:
print("\nFigure 3: Top 10 neurons dose-response curves")

# Identify top 10 neurons by max MN9 activation across all frequencies
neuron_max_activation = {}
for nid in top_200_neurons:
    max_activation = 0
    for freq in FREQ_LIST_FULL:
        if nid in results_exp2_full[freq]:
            max_activation = max(max_activation, results_exp2_full[freq][nid]['mean'])
    neuron_max_activation[nid] = max_activation

# Get top 10
top_10_neurons = sorted(neuron_max_activation.items(), key=lambda x: x[1], reverse=True)[:10]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

for i, (nid, max_rate) in enumerate(top_10_neurons):
    rates = [results_exp2_full[freq][nid]['mean'] for freq in FREQ_LIST_FULL]
    stds = [results_exp2_full[freq][nid]['std'] for freq in FREQ_LIST_FULL]
    
    ax.errorbar(FREQ_LIST_FULL, rates, yerr=stds,
                marker='o', markersize=6, capsize=4, linewidth=1.5, alpha=0.7,
                label=f'Top {i+1} (max {max_rate:.1f} Hz)')

ax.set_xlabel('Activation Frequency (Hz)', fontsize=12)
ax.set_ylabel('Predicted MN9 Firing Rate (Hz)', fontsize=12)
ax.set_title('Top 10 Neurons: MN9 Activation Dose-Response', fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig(fig_dir / 'fig3_top10_response_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"  ✅ Saved: fig3_top10_response_curves.png")

# Print top 10 details
print(f"\n  Top 10 neurons by max MN9 activation:")
for i, (nid, max_rate) in enumerate(top_10_neurons, 1):
    rates_str = ", ".join([f"{results_exp2_full[f][nid]['mean']:.1f}" for f in FREQ_LIST_FULL])
    print(f"    {i:2d}. {nid}: [{rates_str}] Hz")

## 12. Save Complete Results

In [ ]:
print("\n" + "=" * 70)
print("Saving Complete Results")
print("=" * 70)

# Save sufficient neurons (for Exp3 Venn diagram)
# Use 200Hz as the reference (highest activation)
sufficient_neurons_array = np.array(list(sufficient_200hz))
np.save(output_dir / 'sufficient_neurons.npy', sufficient_neurons_array)

print(f"\n✅ Saved sufficient neurons:")
print(f"  {output_dir / 'sufficient_neurons.npy'}")
print(f"  Count: {len(sufficient_neurons_array)} (at 200Hz)")

# Save complete results package
results_package = {
    'results': results_exp2_full,
    'neurons_tested': top_200_neurons,
    'freq_list': FREQ_LIST_FULL,
    'sufficient_neurons': {
        'by_frequency': sufficient_by_freq,
        'any_frequency': list(sufficient_any),
        'at_200hz': list(sufficient_200hz),
    },
    'parameters': {
        'n_trials': N_TRIALS,
        'n_workers': N_WORKERS,
        'mn9_id': NEU_MN9_RIGHT[0],
    },
    'metadata': {
        'date': pd.Timestamp.now().isoformat(),
        'connectome_version': 783,
    }
}

with open(output_dir / 'exp2_complete_results.pkl', 'wb') as f:
    pickle.dump(results_package, f)

print(f"\n✅ Complete package saved:")
print(f"  {output_dir / 'exp2_complete_results.pkl'}")

# Save summary CSV
df_summary.to_csv(output_dir / 'summary.csv', index=False)
print(f"  {output_dir / 'summary.csv'}")

# Save MN9 activation matrix (neurons × frequencies)
mn9_matrix = pd.DataFrame(
    index=top_200_neurons,
    columns=FREQ_LIST_FULL
)

for freq in FREQ_LIST_FULL:
    for nid in top_200_neurons:
        if nid in results_exp2_full[freq]:
            mn9_matrix.loc[nid, freq] = results_exp2_full[freq][nid]['mean']

mn9_matrix.fillna(0).to_csv(output_dir / 'mn9_activation_matrix.csv')
print(f"  {output_dir / 'mn9_activation_matrix.csv'}")
print(f"    (200 neurons × 8 frequencies, MN9 firing rates)")

## 13. Experiment Summary

In [ ]:
print("\n" + "=" * 70)
print("✅ Experiment 2 Complete!")
print("=" * 70)

print(f"\n[Summary Table]")
print(df_summary.to_string(index=False))

print(f"\n[Key Findings]")
print(f"  Neurons tested: {len(top_200_neurons)}")
print(f"  Frequencies tested: {len(FREQ_LIST_FULL)}")
print(f"  Total conditions: {len(top_200_neurons) * len(FREQ_LIST_FULL)}")
print(f"  Sufficient at ANY frequency: {len(sufficient_any)} neurons")
print(f"  Sufficient at 200 Hz: {len(sufficient_200hz)} neurons")

# Most effective frequency
max_sufficient_freq = max(sufficient_by_freq.items(), key=lambda x: len(x[1]))
print(f"  Most effective frequency: {max_sufficient_freq[0]} Hz ({len(max_sufficient_freq[1])} sufficient neurons)")

print(f"\n[Outputs]")
print(f"  Results directory: {output_dir}/")
print(f"  - exp2_complete_results.pkl (full data)")
print(f"  - sufficient_neurons.npy (for Exp3 Venn diagram)")
print(f"  - summary.csv (table)")
print(f"  - mn9_activation_matrix.csv (200 neurons × 8 frequencies)")
print(f"  - figures/ (3 plots)")

print(f"\n[Next Steps]")
print(f"  1. Use sufficient_neurons.npy in Exp3 for Venn diagram")
print(f"  2. Compare with original paper Figure 1e")
print(f"  3. Analyze neuron types using classification data")

print("\n" + "=" * 70)

---

## Appendix: Detailed Analysis

### A1. Frequency-dependent sufficiency

In [ ]:
# Plot: Number of sufficient neurons vs frequency
fig, ax = plt.subplots(figsize=(10, 6))

freqs = sorted(sufficient_by_freq.keys())
counts = [len(sufficient_by_freq[f]) for f in freqs]

bars = ax.bar(range(len(freqs)), counts, color='steelblue',
              edgecolor='black', alpha=0.8)

ax.set_xlabel('Activation Firing Rate (Hz)', fontsize=12)
ax.set_ylabel('Number of Sufficient Neurons', fontsize=12)
ax.set_title('Frequency Dependence of Sufficiency', fontsize=13, fontweight='bold')

ax.set_xticks(range(len(freqs)))
ax.set_xticklabels(freqs)

# Add value labels
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(count), ha='center', va='bottom', fontsize=10)

ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
# plt.savefig(fig_dir / 'appendix_sufficiency_vs_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSufficient neurons per frequency:")
for freq, count in zip(freqs, counts):
    print(f"  {freq:3d} Hz: {count:3d} neurons ({100*count/200:.1f}%)")

### A2. Distribution of MN9 activation strengths

In [ ]:
# Multi-panel histogram for all frequencies
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, freq in zip(axes, FREQ_LIST_FULL):
    mn9_rates = [stats['mean'] for stats in results_exp2_full[freq].values()]
    
    # Only plot active neurons (>0)
    active_rates = [r for r in mn9_rates if r > 0]
    
    if active_rates:
        bins = np.linspace(0, max(active_rates)*1.1, 20)
        ax.hist(active_rates, bins=bins, edgecolor='black', alpha=0.7, color='steelblue')
    
    ax.set_xlabel('MN9 Rate (Hz)', fontsize=10)
    ax.set_ylabel('Neuron Count', fontsize=10)
    ax.set_title(f'{freq} Hz', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add statistics
    n_active = len(active_rates)
    mean_active = np.mean(active_rates) if active_rates else 0
    textstr = f'n={n_active}\nmean={mean_active:.1f}'
    ax.text(0.98, 0.98, textstr, transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('MN9 Activation Distribution Across Frequencies', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
# plt.savefig(fig_dir / 'appendix_activation_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"  ✅ Saved: appendix_activation_distributions.png")

### A3. Top 20 most sufficient neurons (detailed)

In [ ]:
print("\n" + "=" * 70)
print("Top 20 Most Sufficient Neurons (Detailed)")
print("=" * 70)

# Get top 20 by max activation
top_20_sufficient = sorted(neuron_max_activation.items(), key=lambda x: x[1], reverse=True)[:20]

print(f"\n{'Rank':<6}{'Neuron ID':<20}{'Max MN9 Hz':<12}{'At Freq':<10}")
print("="*50)

for i, (nid, max_rate) in enumerate(top_20_sufficient, 1):
    # Find frequency with max activation
    freq_max = max(FREQ_LIST_FULL, 
                   key=lambda f: results_exp2_full[f][nid]['mean'])
    
    print(f"{i:<6}{nid:<20}{max_rate:<12.1f}{freq_max:<10}")

# Save as CSV
df_top20 = pd.DataFrame([
    {
        'Rank': i,
        'Neuron_ID': nid,
        'Max_MN9_Hz': max_rate,
        'At_Frequency': max(FREQ_LIST_FULL, key=lambda f: results_exp2_full[f][nid]['mean'])
    }
    for i, (nid, max_rate) in enumerate(top_20_sufficient, 1)
])
df_top20.to_csv(output_dir / 'top_20_sufficient_neurons.csv', index=False)

print(f"\n✅ Saved: top_20_sufficient_neurons.csv")

### A4. Checkpoint statistics

In [ ]:
# Final checkpoint status
ckpt_final = CheckpointManager(CHECKPOINT_DIR)

print("\n" + "=" * 70)
print("Checkpoint Statistics")
print("=" * 70)

print(f"\n  Total completed tasks: {len(ckpt_final.completed)}")
print(f"  Expected: {len(top_200_neurons) * len(FREQ_LIST_FULL)}")

if len(ckpt_final.completed) == len(top_200_neurons) * len(FREQ_LIST_FULL):
    print(f"\n  ✅ All tasks completed successfully!")
else:
    print(f"\n  ⚠️  Some tasks missing")
    print(f"     Check individual batch outputs above")

# Checkpoint disk usage
total_size = sum(os.path.getsize(f) for f in Path(CHECKPOINT_DIR).glob('*.pkl'))
print(f"\n  Checkpoint disk usage: {total_size/1e6:.1f} MB")
print(f"  Average per task: {total_size/len(ckpt_final.completed)/1024:.1f} KB")

### A5. Memory performance summary

In [ ]:
import psutil

mem = psutil.virtual_memory()
swap = psutil.swap_memory()

print("\n" + "=" * 70)
print("Final Memory Status")
print("=" * 70)
print(f"\n  RAM:  {mem.used/1e9:.2f} GB / {mem.total/1e9:.2f} GB ({mem.percent:.1f}%)")
print(f"  Swap: {swap.used/1e9:.2f} GB")

if swap.used < 0.1e9 and mem.percent < 75:
    print(f"\n  ✅ MEMORY PERFORMANCE: Excellent")
    print(f"     No swap usage, RAM well managed")
elif mem.percent < 85:
    print(f"\n  ✅ MEMORY PERFORMANCE: Good")
else:
    print(f"\n  ⚠️  High memory usage detected")